# 01 - Iceberg Snapshot Time Travel
Every write to an Iceberg table creates a **snapshot**. We can query any past snapshot (time travel) and even roll the live table back to one (revert).

Run the cells top to bottom. If a Nessie URI error appears, change `/api/v2` to `/api/v1` in the config cell and re-run.

## 1. Start Spark (connected to Nessie + MinIO)

In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder.appName("lakehouse-notebook")
    .config("spark.jars.packages",
            "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.5.0,"
            "org.projectnessie.nessie-integrations:nessie-spark-extensions-3.5_2.12:0.77.1,"
            "software.amazon.awssdk:bundle:2.24.8,"
            "software.amazon.awssdk:url-connection-client:2.24.8")
    .config("spark.sql.extensions",
            "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions,"
            "org.projectnessie.spark.extensions.NessieSparkSessionExtensions")
    .config("spark.sql.catalog.nessie", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.nessie.catalog-impl", "org.apache.iceberg.nessie.NessieCatalog")
    .config("spark.sql.catalog.nessie.uri", "http://nessie:19120/api/v2")
    .config("spark.sql.catalog.nessie.ref", "main")
    .config("spark.sql.catalog.nessie.authentication.type", "NONE")
    .config("spark.sql.catalog.nessie.warehouse", "s3://warehouse")
    .config("spark.sql.catalog.nessie.io-impl", "org.apache.iceberg.aws.s3.S3FileIO")
    .config("spark.sql.catalog.nessie.s3.endpoint", "http://minio:9000")
    .config("spark.sql.catalog.nessie.s3.path-style-access", "true")
    .config("spark.sql.catalog.nessie.s3.access-key-id", "minioadmin")
    .config("spark.sql.catalog.nessie.s3.secret-access-key", "minioadmin")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print("Spark ready:", spark.version)

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
org.apache.iceberg#iceberg-spark-runtime-3.5_2.12 added as a dependency
org.projectnessie.nessie-integrations#nessie-spark-extensions-3.5_2.12 added as a dependency
software.amazon.awssdk#bundle added as a dependency
software.amazon.awssdk#url-connection-client added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-10fd2ee6-61b5-4852-ac93-7c6b854eff73;1.0
	confs: [default]
	found org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.5.0 in central
	found org.projectnessie.nessie-integrations#nessie-spark-extensions-3.5_2.12;0.77.1 in central
	found software.amazon.awssdk#bundle;2.24.8 in central
	found software.amazon.awssdk#url-connection-client;2.24.8 in central
	found software.amazon.awssdk#utils;2.24.8 in central
	found org.reactivestreams#reactive-streams;1.0.4 in central
	found software.amazon.awssdk#annotations;2.24.8 in central
	found org.slf4j#slf4j-

Spark ready: 3.5.2


In [2]:
TABLE = "nessie.demo.card_txns"

## 2. Snapshot 1 - initial load (3 approved txns)

In [3]:
spark.sql("CREATE NAMESPACE IF NOT EXISTS nessie.demo")
spark.sql(f"DROP TABLE IF EXISTS {TABLE}")
spark.sql(f"""
    CREATE TABLE {TABLE} (
        transaction_id STRING, card_id STRING, amount DOUBLE,
        currency STRING, txn_status STRING
    ) USING iceberg
""")
spark.sql(f"""
    INSERT INTO {TABLE} VALUES
    ('TXN1','CARD001',120.50,'CHF','APPROVED'),
    ('TXN2','CARD002', 75.00,'EUR','APPROVED'),
    ('TXN3','CARD003',240.00,'CHF','APPROVED')
""")
spark.sql(f"SELECT * FROM {TABLE} ORDER BY transaction_id").show()

SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.


+--------------+-------+------+--------+----------+
|transaction_id|card_id|amount|currency|txn_status|
+--------------+-------+------+--------+----------+
|          TXN1|CARD001| 120.5|     CHF|  APPROVED|
|          TXN2|CARD002|  75.0|     EUR|  APPROVED|
|          TXN3|CARD003| 240.0|     CHF|  APPROVED|
+--------------+-------+------+--------+----------+



## 3. Snapshot 2 - a *bad* bulk update (zeroes all amounts)

In [4]:
spark.sql(f"UPDATE {TABLE} SET amount = 0.0")
spark.sql(f"SELECT * FROM {TABLE} ORDER BY transaction_id").show()

+--------------+-------+------+--------+----------+
|transaction_id|card_id|amount|currency|txn_status|
+--------------+-------+------+--------+----------+
|          TXN1|CARD001|   0.0|     CHF|  APPROVED|
|          TXN2|CARD002|   0.0|     EUR|  APPROVED|
|          TXN3|CARD003|   0.0|     CHF|  APPROVED|
+--------------+-------+------+--------+----------+



## 4. Snapshot 3 - delete TXN3

In [5]:
spark.sql(f"DELETE FROM {TABLE} WHERE transaction_id = 'TXN3'")
spark.sql(f"SELECT * FROM {TABLE} ORDER BY transaction_id").show()

+--------------+-------+------+--------+----------+
|transaction_id|card_id|amount|currency|txn_status|
+--------------+-------+------+--------+----------+
|          TXN1|CARD001|   0.0|     CHF|  APPROVED|
|          TXN2|CARD002|   0.0|     EUR|  APPROVED|
+--------------+-------+------+--------+----------+



## 5. Snapshot history

In [6]:
snaps = spark.sql(f"SELECT snapshot_id, committed_at, operation FROM {TABLE}.snapshots ORDER BY committed_at")
snaps.show(truncate=False)
first_snapshot = snaps.collect()[0]["snapshot_id"]
print("First (clean) snapshot id =", first_snapshot)

+-------------------+-----------------------+---------+
|snapshot_id        |committed_at           |operation|
+-------------------+-----------------------+---------+
|4520301994001698324|2026-08-01 22:21:21.987|append   |
|4347279833063681456|2026-08-01 22:23:20.559|overwrite|
|8940656233256373808|2026-08-01 22:23:49.632|overwrite|
+-------------------+-----------------------+---------+

First (clean) snapshot id = 4520301994001698324


## 6. Time travel - read the past (table itself unchanged)

In [7]:
spark.sql(f"SELECT * FROM {TABLE} VERSION AS OF {first_snapshot} ORDER BY transaction_id").show()

+--------------+-------+------+--------+----------+
|transaction_id|card_id|amount|currency|txn_status|
+--------------+-------+------+--------+----------+
|          TXN1|CARD001| 120.5|     CHF|  APPROVED|
|          TXN2|CARD002|  75.0|     EUR|  APPROVED|
|          TXN3|CARD003| 240.0|     CHF|  APPROVED|
+--------------+-------+------+--------+----------+



## 7. Rollback - make the live table become that past state

In [8]:
spark.sql(f"CALL nessie.system.rollback_to_snapshot(table => 'demo.card_txns', snapshot_id => {first_snapshot})")
print("Live table after rollback:")
spark.sql(f"SELECT * FROM {TABLE} ORDER BY transaction_id").show()

Live table after rollback:
+--------------+-------+------+--------+----------+
|transaction_id|card_id|amount|currency|txn_status|
+--------------+-------+------+--------+----------+
|          TXN1|CARD001| 120.5|     CHF|  APPROVED|
|          TXN2|CARD002|  75.0|     EUR|  APPROVED|
|          TXN3|CARD003| 240.0|     CHF|  APPROVED|
+--------------+-------+------+--------+----------+



The rollback is itself a new snapshot - nothing is ever lost.

In [9]:
spark.sql(f"SELECT snapshot_id, committed_at, operation FROM {TABLE}.snapshots ORDER BY committed_at").show(truncate=False)

+-------------------+-----------------------+---------+
|snapshot_id        |committed_at           |operation|
+-------------------+-----------------------+---------+
|4520301994001698324|2026-08-01 22:21:21.987|append   |
|4347279833063681456|2026-08-01 22:23:20.559|overwrite|
|8940656233256373808|2026-08-01 22:23:49.632|overwrite|
+-------------------+-----------------------+---------+

